# Energy Regression Model — SolarSense AR

Trains a GradientBoosting regression model to predict monthly solar energy output.
Features: panel count, system kW, irradiance, roof tilt, shadow loss %  
Target: monthly kWh output  
Export: ONNX format for FastAPI inference


In [ ]:
# Install dependencies
# !pip install scikit-learn numpy pandas matplotlib skl2onnx onnx requests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

In [ ]:
# ── Synthetic dataset (replace with real PVGIS CSV when available) ──────────
# Feature columns: panel_count, capacity_kw, irradiance, tilt_deg, shadow_pct, perf_ratio
np.random.seed(42)
n = 5000

panel_count = np.random.randint(2, 50, n)
capacity_kw = panel_count * 0.55
irradiance = np.random.uniform(4.5, 6.5, n)  # kWh/m²/day
tilt_deg = np.random.choice([0, 15, 25, 30, 45], n)  # rooftop angles
shadow_pct = np.random.uniform(0, 20, n)  # % shading loss
perf_ratio = np.random.uniform(0.70, 0.85, n)
is_sloped = (tilt_deg > 0).astype(float)

# Target: monthly kWh (physics-based formula + noise)
tilt_factor = 1 + 0.05 * is_sloped
monthly_kwh = (
    capacity_kw * irradiance * 30 * perf_ratio * tilt_factor
    * (1 - shadow_pct / 100)
    + np.random.normal(0, 15, n)  # measurement noise
).clip(0)

df = pd.DataFrame({
    'panel_count': panel_count,
    'capacity_kw': capacity_kw,
    'irradiance': irradiance,
    'tilt_deg': tilt_deg,
    'shadow_pct': shadow_pct,
    'perf_ratio': perf_ratio,
    'is_sloped': is_sloped,
    'monthly_kwh': monthly_kwh
})

print(f'Dataset: {len(df)} samples')
df.describe()

In [ ]:
# ── Train/test split (temporal-style: last 20% as test) ──────────────────────
FEATURES = ['panel_count', 'capacity_kw', 'irradiance', 'tilt_deg', 'shadow_pct', 'perf_ratio', 'is_sloped']
TARGET = 'monthly_kwh'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# ── Train GradientBoosting pipeline ──────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.08,
        subsample=0.8,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

print(f'MAE:  {mae:.2f} kWh')
print(f'R²:   {r2:.4f}')
print(f'MAPE: {mape:.2f}%  (target: <10%)')

In [ ]:
# ── Cross-validation ─────────────────────────────────────────────────────────
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
print(f'CV R² scores: {cv_scores.round(4)}')
print(f'Mean CV R²:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Feature importance plot ───────────────────────────────────────────────────
importances = pipeline.named_steps['model'].feature_importances_
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(FEATURES, importances, color='#F5A623')
ax.set_xlabel('Feature Importance')
ax.set_title('Energy Regression — Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150)
plt.show()

In [ ]:
# ── Actual vs predicted scatter ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, color='#1B2A4A', s=10)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual kWh/month')
ax.set_ylabel('Predicted kWh/month')
ax.set_title(f'Energy Regression — R² = {r2:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnx

    initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
    onnx_model = convert_sklearn(pipeline, initial_types=initial_type, target_opset=17)

    with open('../backend/app/ml/energy_regressor.onnx', 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print('✓ ONNX model saved to backend/app/ml/energy_regressor.onnx')
except ImportError:
    print('skl2onnx not installed — skipping ONNX export (run: pip install skl2onnx)')